In [1]:
# run in terminal to mount the network drive in WSL
# sudo mount -t drvfs '\\data03\pi-vannoort' /mnt/data03 -o username=noort

In [8]:
import os
from pathlib import Path
import pandas as pd
import pysam
from icecream import ic
import numpy as np
from tqdm import tqdm

import sys
from pathlib import Path
# Add nuctool directory to path for imports
nuctool_path = Path.cwd().parent / 'nuctool'
if str(nuctool_path) not in sys.path:
    sys.path.insert(0, str(nuctool_path))
from Plotter import Plotter



In [9]:
filename = Path(r"/mnt/data03/Noort/Data/Daxinger/dnmt.bam")
filename = Path(r"/mnt/data03/Gert-Jan/PacBio_data/Gal-locus-6mA-only/gal_locus.Raf_6mA.bam")

if not filename.with_suffix(".bai").exists() and filename.exists():
    pysam.index(filename.as_posix())

bam = pysam.AlignmentFile(filename, "rb")

In [10]:
# Select Locus

def extract_locus(locus_txt):
    locus_txt = locus_txt.replace(',', '')
    parts = locus_txt.split(':')
    chrom = parts[0]
    start, end = map(int, parts[1].split('-'))
    return (chrom, start, end)

locus = extract_locus("chrII:278,635-278,757")
locus = extract_locus("chrII:272,114-280,662")
locus = extract_locus("chrII:275,320-277,456")
locus = extract_locus("chrII:275,739-275,805")


ic(locus, locus[-1]-locus[1])


ic| locus: ('chrII', 275739, 275805), locus[-1]-locus[1]: 66


(('chrII', 275739, 275805), 66)

In [11]:
for read in bam.fetch(*locus):
    print("Read:", read.query_name)

    mm = read.get_tag("MM") if read.has_tag("MM") else None
    ml = read.get_tag("ML") if read.has_tag("ML") else None
    
    print("MM:", str(mm)[:100])
    if ml:
        print("ML (first 20):", list(ml[:20]))
    break

Read: m84219_240203_140226_s3/264311055/ccs
MM: C+m,4,2,12,3,5,0,3,11,3,1,0,3,6,2,5,7,2,0,2,0,5,0,0,0,1,13,1,4,7,5,0,11,0,2,4,0,0,8,17,6,9,3,12,1,22
ML (first 20): [102, 23, 71, 58, 48, 17, 140, 44, 22, 206, 73, 142, 37, 202, 115, 27, 0, 32, 9, 110]


In [ ]:
def get_modification_probabilities(read):
    """
    Extracts modification probabilities from MM and ML tags.
    Returns a dict of {mod_type: {ref_pos: probability}}
    """
    if not read.has_tag("MM"):
        return {}

    mm = read.get_tag("MM")
    ml = read.get_tag("ML") if read.has_tag("ML") else []
    
    mods = {}
    ml_idx = 0
    seq = read.query_sequence
    
    # Pre-calculate query to reference mapping once per read
    aligned_pairs = read.get_aligned_pairs()
    q_to_r = {q: r for q, r in aligned_pairs if q is not None}
    
    for section in mm.split(';'):
        if not section: continue
        header, *counts = section.split(',')
        if '+' not in header: continue
        
        target_base = header[0]
        # Find all occurrences of target_base in read sequence
        target_indices = [i for i, b in enumerate(seq) if b == target_base]
        
        current_mod_probs = {}
        target_idx = 0
        
        for c in counts:
            try:
                skip = int(c)
            except ValueError:
                continue
            
            target_idx += skip
            if target_idx < len(target_indices):
                q_pos = target_indices[target_idx]
                r_pos = q_to_r.get(q_pos)
                if r_pos is not None:
                    prob = ml[ml_idx] / 255.0 if ml_idx < len(ml) else 1.0
                    current_mod_probs[r_pos] = prob
                
                ml_idx += 1
                target_idx += 1 

        mods[header] = current_mod_probs
        
    return mods

def get_read_methylation_string(read, chrom, start, end, threshold=0.5):
    """
    Returns a string for the genomic region [start, end) based on a single read.
    """
    mods = get_modification_probabilities(read)
    active_mod_dicts = list(mods.values())

    aligned_pairs = read.get_aligned_pairs()
    seq = read.query_sequence
    locus_length = end - start
    res = ["-"] * locus_length
    
    for q_pos, r_pos in aligned_pairs:
        if r_pos is not None and start <= r_pos < end:
            idx = r_pos - start
            if q_pos is not None:
                base = seq[q_pos].lower()
                
                max_prob = 0
                for m_dict in active_mod_dicts:
                    max_prob = max(max_prob, m_dict.get(r_pos, 0))
                
                if max_prob > threshold:
                    base = base.upper()
                
                res[idx] = base
            
    return "".join(res)

n_reads = 300 

bam = pysam.AlignmentFile(filename, "rb")
chrom, start, end = locus

# Get a stable consensus sequence
reference_sequence_lookup = ["?"] * (end - start)
for read in bam.fetch(*locus):
    aligned_pairs = read.get_aligned_pairs()
    seq = read.query_sequence
    for q_pos, r_pos in aligned_pairs:
        if r_pos is not None and start <= r_pos < end and q_pos is not None:
            reference_sequence_lookup[r_pos - start] = seq[q_pos].lower()
    if "?" not in reference_sequence_lookup:
        break

meth_fwd = np.zeros(end - start)
meth_rev = np.zeros(end - start)
cov_fwd = np.zeros(end - start)
cov_rev = np.zeros(end - start)

print(f"Locus: {chrom}:{start}-{end}")
# Use list() or similar to identify total reads if possible, otherwise stick to fetch
reads_list = list(bam.fetch(*locus))
total_available = len(reads_list)
n_to_process = min(n_reads, total_available)

for i in tqdm(range(n_to_process)):
    read = reads_list[i]
    meth_string = get_read_methylation_string(read, *locus)
    
    for idx, char in enumerate(meth_string):
        if char != '-':
            if read.is_reverse:
                cov_rev[idx] += 1
                if char.isupper(): meth_rev[idx] += 1
            else:
                cov_fwd[idx] += 1
                if char.isupper(): meth_fwd[idx] += 1

# Compute fractions
f_meth_fwd = meth_fwd / np.where(cov_fwd > 0, cov_fwd, 1)
f_meth_rev = meth_rev / np.where(cov_rev > 0, cov_rev, 1)

# Plotting
plotter = Plotter(fig_size=(15, 6))
plotter.new(nrows=2, ncols=1)
ax1, ax2 = plotter.axes

x_coords = np.arange(start, end)
ax1.bar(x_coords, f_meth_fwd, color='skyblue', label='Forward Reads', width=1.0)
ax1.set_ylabel("FWD Meth")
ax1.set_ylim(0, 1)
ax1.set_xlim(start, end)

ax2.bar(x_coords, f_meth_rev, color='salmon', label='Reverse Reads', width=1.0)
if len(x_coords) < 200:
    ax2.set_xticks(x_coords)
    ax2.set_xticklabels(reference_sequence_lookup, fontfamily='monospace', fontsize=7)

ax2.set_ylabel("REV Meth")
ax2.set_ylim(0, 1)
ax2.set_xlim(start, end)

plotter.caption(f"Methylation yield from {filename.name}.")

Locus: chrII:275739-275805


  1%|▏         | 4/300 [00:53<59:34, 12.08s/it]  